# Zebra DS457 barcode scanner — connect & scan test

Scanner in **USB CDC** mode: on Linux it appears as `/dev/ttyACM0` and streams
each decoded barcode as plain text. Far simpler than HID keyboard mode — just a
serial port: no grab, no keymap, no window focus.

Key rule: a `timeout` or `disconnected` result is **not** an empty barcode —
always branch on `status` / `connected`.

No sudo needed: your user is in the `dialout` group.

## Setup

In [ ]:
from ds457_driver import DS457

# set your port:  "COM15" on Windows,  "/dev/ttyACM0" on Linux
scanner = DS457(port="/dev/ttyACM0")

## Connect

In [ ]:
print("connect()        :", scanner.connect())
print("check_connection :", scanner.check_connection())
if scanner.is_connected():
    print("port             :", scanner.port)

## Scan one barcode

Run the cell, then present a barcode to the fixed-mount imager within 10 s.

In [ ]:
r = scanner.scan(timeout=10.0)
print("status:", r.status)
print("data  :", r.data)

## Continuous scanning

Read several barcodes in a row. Stops after 5 scans or a 15 s idle timeout.

In [ ]:
count = 0
while count < 5:
    r = scanner.scan(timeout=15.0)
    if not r.connected:
        print("scanner DISCONNECTED")
        break
    if r.status == "timeout":
        print("(idle — stopping)")
        break
    count += 1
    print(f"#{count}: {r.data}")

## Disconnect handling

Unplug the scanner's USB and re-run this cell — the status flips to
`disconnected` instead of returning a blank scan.

In [ ]:
r = scanner.scan(timeout=5.0)

if not r.connected:
    print("scanner is DISCONNECTED — do not trust any data")
elif r.status == "timeout":
    print("no barcode presented in time")
else:
    print("scanned:", r.data)

## Close

Closes the serial port.

In [ ]:
scanner.close()
print("closed:", not scanner.is_connected())